## Запуск Flask из Jupyter Notebook

### Порядок действий

1. Откройте папку **`ffmpeg-stream-rec`** как корень workspace в Cursor и ядро **`Python (ffmpeg-stream-rec .venv)`**.
2. Выполните ячейку **«Запуск сервера»** — после проверки HTTP **откроется браузер** с приложением (если система разрешает `webbrowser`).
3. Чтобы **остановить только сервер** (освободить порт), выполните ячейку **«Остановка сервера»**.
4. Чтобы **сбросить всё состояние ядра** и гарантированно убить потоки — ячейка **«Перезапуск ядра»** (или меню Notebook → Restart Kernel).

### Если браузер не открылся

Перейдите вручную по адресу из вывода ячейки (обычно http://127.0.0.1:5000 ).

### Занят порт

Перед запуском Jupyter: `export FLASK_RUN_PORT=5001` или поменяйте `PORT` в ячейке запуска.

### Запуск сервера

In [ ]:
import os
import sys
import threading
import time
import webbrowser
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

from IPython.display import HTML, display

# Глобальное состояние dev-сервера (shutdown через make_server)
_ffmpeg_rec_dev = {"srv": None, "thread": None}


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "recorder" / "__init__.py").is_file():
            return base
    raise FileNotFoundError(
        "Не найден пакет recorder/. Откройте репозиторий ffmpeg-stream-rec как папку workspace."
    )


ROOT = find_repo_root()
os.chdir(ROOT)
_rs = str(ROOT)
if _rs not in sys.path:
    sys.path.insert(0, _rs)

HOST = os.environ.get("FLASK_RUN_HOST", "127.0.0.1")
PORT = int(os.environ.get("FLASK_RUN_PORT", "5000"))

# После успешного старта открыть вкладку в браузере
OPEN_BROWSER = True

print("Рабочая директория:", ROOT)
print("Python:", sys.executable)


def start_dev_server(*, open_browser: bool = True) -> None:
    from werkzeug.serving import make_server

    from recorder import create_app

    if _ffmpeg_rec_dev["thread"] and _ffmpeg_rec_dev["thread"].is_alive():
        url = f"http://{HOST}:{PORT}/"
        print(f"Сервер уже запущен → {url}")
        if open_browser:
            webbrowser.open(url)
            print("Запрошено открытие в браузере.")
        display(HTML(f'<p><a href="{url}" target="_blank"><b>{url}</b></a></p>'))
        return

    app = create_app()
    try:
        srv = make_server(HOST, PORT, app, threaded=True)
    except OSError as e:
        print(f"Не удалось занять {HOST}:{PORT} — {e}")
        return

    _ffmpeg_rec_dev["srv"] = srv

    def run() -> None:
        try:
            srv.serve_forever()
        except Exception as e:
            print(f"[сервер] {e!r}")

    th = threading.Thread(target=run, daemon=True, name="flask-dev")
    _ffmpeg_rec_dev["thread"] = th
    th.start()
    time.sleep(0.6)

    if not th.is_alive():
        print("Поток сервера завершился сразу — см. сообщения выше.")
        _ffmpeg_rec_dev["srv"] = None
        _ffmpeg_rec_dev["thread"] = None
        return

    url = f"http://{HOST}:{PORT}/"
    try:
        with urlopen(url, timeout=5) as resp:
            print(f"Проверка: главная страница отвечает HTTP {resp.status}")
    except URLError as e:
        print("Пока не удалось подключиться:", e)

    print(f"\nПриложение: {url}\n")
    display(HTML(f'<p><a href="{url}" target="_blank"><b>Открыть в браузере</b></a></p>'))

    if open_browser:
        opened = webbrowser.open(url)
        print(
            "Браузер открыт автоматически."
            if opened
            else "Не удалось открыть браузер автоматически — перейдите по ссылке выше."
        )


def stop_dev_server() -> None:
    srv = _ffmpeg_rec_dev.get("srv")
    th = _ffmpeg_rec_dev.get("thread")
    if srv is None and (th is None or not th.is_alive()):
        print("Сервер не запущен (или уже остановлен).")
        return
    try:
        srv.shutdown()
    except Exception as e:
        print("shutdown:", e)
    if th is not None and th.is_alive():
        th.join(timeout=15)
    _ffmpeg_rec_dev["srv"] = None
    _ffmpeg_rec_dev["thread"] = None
    print("Локальный сервер остановлен (порт должен быть свободен).")


start_dev_server(open_browser=OPEN_BROWSER)

### Остановка сервера

Корректно закрывает HTTP-сервер Werkzeug в этом ядре. Активные записи FFmpeg при этом **не останавливает** — ими управляет веб-приложение или отдельные процессы.

In [ ]:
stop_dev_server()

### Перезапуск ядра Jupyter

Полностью перезапускает kernel (как **Notebook: Restart Kernel**): обнуляются переменные, останавливаются потоки. После этого снова выполните ячейку **«Запуск сервера»**.

Если ячейка не сработала в вашей среде — используйте палитру команд: **Notebook: Restart Kernel**.

In [ ]:
from IPython import get_ipython

_ip = get_ipython()
if _ip is None or not hasattr(_ip, "kernel"):
    print("Не удалось получить kernel — перезапустите ядро через меню Cursor.")
else:
    print("Перезапуск kernel…")
    _ip.kernel.do_shutdown(restart=True)